In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
v1

array([ 2.13903859e-02, -7.39799738e-02,  1.42069475e-03,  2.13816315e-02,
        2.45113391e-02,  3.15582603e-02, -1.10839710e-01, -1.05017453e-01,
       -6.18258938e-02, -6.42312970e-03,  3.72394198e-03,  9.06393528e-02,
       -9.49937198e-03,  6.53976947e-02,  1.10946735e-02, -2.10097395e-02,
       -3.35125700e-02, -4.31677736e-02,  9.96346213e-03,  1.41969677e-02,
       -6.40415177e-02, -7.04179332e-03, -7.91188180e-02,  5.80030978e-02,
        1.30213588e-03,  4.19733720e-03,  5.70979156e-02,  6.39447868e-02,
        2.49903090e-02, -3.95876467e-02, -3.79506387e-02,  2.70394497e-02,
        1.79423485e-02,  1.72272474e-02,  3.43311429e-02,  9.29058157e-03,
        5.86054958e-02, -4.97789867e-02, -5.05369157e-03,  4.34328243e-02,
       -1.56622920e-02, -2.97534615e-02, -5.13324142e-03,  5.13414778e-02,
        6.16064621e-03,  6.86980411e-02, -1.29505135e-02, -5.61938696e-02,
       -1.08265216e-02,  5.96683659e-02,  5.29939905e-02, -3.42754982e-02,
       -4.15274203e-02, -

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
dv

array([-4.81206924e-02, -1.29942119e-01, -4.86811250e-03,  1.37495529e-02,
       -8.51096865e-03,  6.97841048e-02, -7.55671831e-03,  4.09572013e-02,
       -1.02165632e-01, -3.45050506e-02,  2.85521504e-02, -5.81430867e-02,
       -1.85113158e-02, -3.17839384e-02,  5.29631525e-02,  3.96292955e-02,
       -9.84594040e-03, -6.45324588e-02,  7.05098137e-02, -6.39589503e-02,
       -3.55156884e-02, -1.75489970e-02, -3.58401798e-02,  5.50819037e-04,
        3.38445716e-02,  2.91762855e-02,  3.24278660e-02, -5.64806117e-03,
        8.42592865e-02,  5.90306148e-02,  2.08772346e-02, -5.44836652e-03,
        2.98404153e-02,  1.30820405e-02,  6.36078045e-02, -3.06394044e-02,
        6.28459752e-02, -1.69217717e-02, -4.65758033e-02, -5.13761863e-02,
       -3.81561220e-02, -7.01197013e-02, -9.55517292e-02,  5.01640178e-02,
        8.57913494e-03,  1.19207380e-02, -1.40597159e-02, -7.98979774e-03,
       -3.44238728e-02, -2.34269779e-02,  2.89466027e-02,  2.68904790e-02,
       -4.87408489e-02, -

In [6]:
v1.dot(dv)

np.float32(0.32332397)

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)

np.float32(0.019730574)

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
from tqdm.auto import tqdm
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [12]:
vectors[10].shape

(384,)

In [13]:
import numpy as np
X = np.array(vectors)

In [14]:
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.01099552, -0.11074744, -0.02536942, ...,  0.09022228,
        -0.02697371,  0.01975672],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [-0.03652925,  0.01415426, -0.06838644, ...,  0.04316786,
         0.08105537, -0.02148626],
       [-0.13091588, -0.06990605, -0.0093188 , ..., -0.00044342,
        -0.0128573 ,  0.01426918],
       [-0.07984784,  0.01926981,  0.02544978, ..., -0.03368027,
        -0.01884026,  0.05837054]], shape=(1350, 384), dtype=float32)

In [15]:
scores = X.dot(v1)

In [16]:
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

In [17]:
idx = np.argmax(scores)
idx, scores[idx]
# Return the document with the highest score

(np.int64(2), np.float32(0.762941))

In [18]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [19]:
top5 =np.argsort(scores)[-5:]
top5 = top5[::-1]
top5
# Indexes of the top 5 vectors with the highest scores related to the query vector v1

array([  2, 625, 907, 538,   7])

In [20]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [22]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [23]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [24]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [25]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [26]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [27]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [28]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [29]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

In [30]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [31]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [32]:
vector_assistant.rag("the program has already started, can I still sign up?")

'Yes. You can still join even if the program has already started. If you want a certificate, make sure to submit your project while submissions are still open.'